In [1]:
import pandas as pd
from pathlib import Path
import duckdb
from openpyxl.writer.excel import ExcelWriter
from scripts.h2py import ignores

In [2]:
file_path = Path(r"E:\online-order-processing\samples\shopee\result_test.xlsx")

In [3]:
source_df = pd.read_excel(file_path, sheet_name="source")
split_df = pd.read_excel(file_path, sheet_name="split")
merge_df = pd.read_excel(file_path, sheet_name="merge")

In [4]:
merge_df.columns

Index(['STT', 'Mã đơn hàng', 'Ngày đặt hàng', 'Mã vận đơn',
       'Đơn Vị Vận Chuyển', 'Tên sản phẩm', 'Tên phân loại hàng',
       'Giá ưu đãi', 'Số lượng', 'Tổng giá bán (sản phẩm)',
       'Tong gia tri don hang', 'Mã giảm giá của Shop',
       'Tổng số tiền người mua thanh toán', 'Phí cố định', 'Phí Dịch Vụ',
       'Phí thanh toán', 'DOANH THU', 'Ngày làm đơn hàng', 'Số đơn hàng',
       'Trạng thái', 'Ngày hoàn', 'Code', 'Tên SP', 'Giá', 'Số lượng.1',
       'Thành tiền', 'Chênh lệch'],
      dtype='str')

In [5]:
# Lấy unique "Tên sản phẩm", "Tên phân loại hàng", "Giá ưu đãi"
unique_columns = ["Tên sản phẩm", "Tên phân loại hàng", "Giá ưu đãi"]
source_filtered = source_df[unique_columns].drop_duplicates().sort_values(by="Tên sản phẩm")
merge_filtered = merge_df[unique_columns + ["Code", "Tên SP"]].drop_duplicates().sort_values(by="Tên sản phẩm")

# Join 2 bảng để xem mối quan hệ giữa combo và sản phẩm
join_result = duckdb.sql(f"""
           SELECT DISTINCT * FROM source_filtered sf
           JOIN merge_filtered mf
               USING ("Tên sản phẩm", "Tên phân loại hàng", "Giá ưu đãi")
               ORDER BY ("Tên sản phẩm", "Tên phân loại hàng", "Giá ưu đãi")
           """).to_df()

# Lọc các combo
import duckdb

# Assuming your data is in a DataFrame or table named 'sales_data'
combo_result = duckdb.sql("""
    SELECT
        "Tên sản phẩm",
        "Tên phân loại hàng",
        "Code",
        "Tên SP",
        COUNT(DISTINCT "Tên sản phẩm") AS count
    FROM join_result
    GROUP BY
        "Tên sản phẩm",
        "Tên phân loại hàng",
        "Code",
        "Tên SP"
""").to_df()

In [6]:
with pd.ExcelWriter("Join Result.xlsx") as w:
    join_result.to_excel(w, sheet_name= "Join result", index=False)
    combo_result.to_excel(w, sheet_name="Combo result", index=False)